In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    transforms.Resize((28, 28)),  # TODO: Resize to 28x28
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    transforms.ToTensor(),  # TODO: Convert to Tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])  # TODO: Normalize with ImageNet mean and std
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# (A-Z) so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)


# i disply some images from the dataset to sure the code works well :)
import matplotlib.pyplot as plt
import numpy as np

images, labels = next(iter(train_loader))

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(1, 6, figsize=(12, 3))
for i in range(6):
    img = images[i].numpy().transpose(1, 2, 0)
    img = (img * std) + mean
    img = np.clip(img, 0, 1)

    letter = letters[labels[i].item() - 1]

    axes[i].imshow(img)
    axes[i].set_title(letter)
    axes[i].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Here i define dataloader and split it intn to train and test

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

import matplotlib.pyplot as plt
import numpy as np

# Display some sample images
#same code of plot above
images, labels = next(iter(train_loader))

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(1, 6, figsize=(12, 3))

for i in range(6):
    img = images[i].numpy().transpose(1, 2, 0)
    img = (img * std) + mean
    img = np.clip(img, 0, 1)

    letter = letters[labels[i].item() - 1] # cause make index start from zero  it give me error in training and here he give me out of index before i put it so i put -1

    axes[i].imshow(img)
    axes[i].set_title(letter)
    axes[i].axis("off")

plt.tight_layout()
plt.show()



In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained EfficientNetV2-b0
model = efficientnet_v2_s(pretrained=True)

# Freeze the backbone (feature extractor)
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier head
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 26)  ## 26 classes i have

# Move model to device
model = model.to(device)

print(model)
print(device)



In [ ]:
# Write your code here
import torch
from tqdm import tqdm # Shows progress bar

# Training loop
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train() # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader):
        images = images.to(device)
        labels = labels.to(device)

        # to make index start from 0 not 1 (1 give me error)
        labels = labels - 1

        optimizer.zero_grad()
        outputs = model(images) # Forward pass
        loss = criterion(outputs, labels) # Compute loss

        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        running_loss += loss.item()

        # Track accuracy
        preds = torch.argmax(outputs, dim=1)  # Get class with highest probability
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / len(loader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


# 🔹 Validation Loop
def validate(model, loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            # to make index start from 0 not 1 (1 give me error)
            labels = labels - 1

            outputs = model(images) # Forward pass
            loss = criterion(outputs, labels)  # Compute loss

            running_loss += loss.item()

           # Compute accuracy
            preds = torch.argmax(outputs, dim=1)  # Get predicted class
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = running_loss / len(loader)
    accuracy = 100 * correct / total # Compute accuracy in percentage
    return avg_loss, accuracy



In [ ]:
# Write your code here
import torch.optim as optim
import matplotlib.pyplot as plt

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)

# Adam optimizer only classifier
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

num_epochs = 5  # i put 5 to make it faster there  isnot enough time for 20 :(

# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

# Plot the training and validation losses
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

# Plot the training and validation Accuracy
plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(val_accuracies, label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training vs Validation Accuracy")
plt.legend()

plt.tight_layout()
plt.show()



In [ ]:
# Write your code here

def validate_tta(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

          # to make index start from 0 not 1 (1 give me error)
            labels = labels - 1

            # Predicte original images
            out1 = model(images)

            # Predicte horizontally flipped images
            h_flipped = torch.flip(images, dims=[3])
            out2 = model(h_flipped)

            # Predicte vertically flipped images
            v_flipped = torch.flip(images, dims=[2])
            out3 = model(v_flipped)

            # Average predictions
            outputs = (out1 + out2 + out3) / 3.0

            loss = criterion(outputs, labels) ## compute loss
            running_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)   # Get class with highest probability
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = running_loss / len(loader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


tta_loss, tta_acc = validate_tta(model, test_loader, criterion, device)
print(f"TTA Val Loss: {tta_loss:.4f} | TTA Val Acc: {tta_acc:.2f}%")


